# Module `algorithms/neighborhood.py`

Ce module est la **fondation partagee** par toutes les metaheuristiques VRP-CDR du projet (GRASP, recuit simule, tabou, genetique). Il regroupe :

- la representation `VRPSolution` (dataclass) ;
- les operateurs de **voisinage deterministes** (`relocate_inter`, `swap_inter`, `two_opt`) en best-improvement ;
- la routine `local_search` qui les compose jusqu'a stabilite ;
- les versions aleatoires (`random_*`) utilisees par le recuit simule.

**Pourquoi un module separe ?** Parce que les quatre metaheuristiques utilisent **exactement les memes briques** de transformation de solution. Les dupliquer dans chaque algorithme creerait un risque de divergence (un bug corrige a un endroit mais pas a l'autre) et empecherait toute comparaison equitable des algorithmes : si GRASP et le tabou n'utilisaient pas la meme implementation de 2-opt, on ne saurait jamais si une difference de qualite vient de la strategie d'exploration ou de la qualite des operateurs. La regle transverse du projet est donc : **tout nouvel operateur de voisinage est ajoute ici**, jamais duplique dans un algorithme.

**Notion de voisinage** (au sens metaheuristique). Le voisinage `N(s)` d'une solution `s` est l'ensemble des solutions atteignables en **un mouvement** par un operateur donne. Le choix des operateurs definit la **topologie** de l'espace de recherche : deux solutions distantes en nombre de mouvements peuvent etre tres differentes en cout. Cette notion est centrale dans toute la theorie de la recherche locale (Aarts & Lenstra [1, chap. 1]).

Les trois operateurs implementes (relocate, swap, 2-opt) sont les **operateurs classiques** du VRP, etablis depuis Lin [2] (2-opt) et Or [3] (Or-opt, dont relocate est un cas degenere). Ils ont fait leurs preuves dans la quasi-totalite des heuristiques VRP modernes (Cordeau et al. [4, section 4]).


In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent / "src"))

from cesipath import GraphGenerationConfig, GraphGenerator
from cesipath.algorithms import (
    VRPSolution,
    is_feasible,
    local_search,
    random_neighbor,
    relocate_inter,
    route_cost,
    route_load,
    swap_inter,
    total_cost,
    two_opt,
    two_opt_intra,
)
from cesipath.algorithms.grasp import greedy_randomized_construction
from cesipath.solver_input import build_static_solver_input
import random


## 1. Instance et solution initiale

On fabrique une petite instance puis une solution admissible via le constructeur glouton de GRASP. Cela donne une base concrete sur laquelle manipuler les operateurs.


In [2]:
cfg = GraphGenerationConfig(node_count=12, seed=7)
instance = GraphGenerator(cfg).generate()
si = build_static_solver_input(instance)

rng = random.Random(0)
routes = greedy_randomized_construction(si, rcl_alpha=0.0, rng=rng)

print(f"{len(routes)} sous-tournees, cout initial = {total_cost(routes, si.cost_matrix):.2f}")
for idx, r in enumerate(routes):
    print(f"  r{idx + 1} : {r}  charge={route_load(r, si.demands)}/{si.vehicle_capacity}")


2 sous-tournees, cout initial = 1002.17
  r1 : [0, 2, 11, 1, 7, 3, 5, 4, 9, 0]  charge=40/40
  r2 : [0, 6, 8, 10, 0]  charge=15/40


## 2. Representation `VRPSolution`

`VRPSolution` est une `dataclass` minimale : juste une liste de routes et un cout total. **Pourquoi pas une classe avec des methodes riches ?** Parce que les operateurs de voisinage manipulent directement la structure de routes (`list[list[int]]`) en interne pour des raisons de performance : creer un objet `VRPSolution` a chaque mouvement intermediaire serait du gaspillage. La dataclass sert donc d'**emballage final** retourne par les algorithmes au code appelant - elle ne circule pas dans les boucles internes.

C'est un choix de **separation de responsabilites** : la couche operateurs travaille sur `Routes = list[list[int]]` (representation primitive, rapide a copier), la couche algorithme renvoie `VRPSolution` (representation utilisateur, immuable sauf via une nouvelle instance). La propriete `route_count` est un raccourci sur `len(routes)` pour eviter d'avoir a connaitre la structure interne.


In [3]:
sol = VRPSolution(routes=routes, total_cost=round(total_cost(routes, si.cost_matrix), 2))
print(sol)
print("nombre de tournees :", sol.route_count)


VRPSolution(routes=[[0, 2, 11, 1, 7, 3, 5, 4, 9, 0], [0, 6, 8, 10, 0]], total_cost=1002.17)
nombre de tournees : 2


## 3. Cout de tournee, charge, admissibilite

Trois fonctions atomiques que tous les operateurs reutilisent :

- **`route_cost`** : somme les couts des aretes consecutives (depot -> clients -> depot). Lineaire en la taille de la route. C'est ici qu'entre en jeu la **fermeture metrique** (voir `metric_closure.ipynb`) : on n'utilise jamais le cout brut du graphe, mais la matrice `cost_matrix` precalculee qui contient les plus courts chemins entre toute paire de noeuds. C'est ce qui garantit l'inegalite triangulaire et donc la coherence de tous les calculs `delta` differentiels des operateurs.
- **`route_load`** : somme les demandes des **clients** uniquement (on exclut explicitement les depots aux extremites via `route[1:-1]`). C'est la charge transportee par le vehicule sur cette tournee.
- **`is_feasible`** : verifie deux invariants structurels - chaque route commence et finit au depot, et la charge ne depasse pas la capacite. **Limite importante** : cette fonction ne valide **pas** les contraintes metier du VRP-CDR (arcs interdits via `EdgeStatus.FORBIDDEN`, couverture totale des clients, conformite au snapshot dynamique). Ces validations sont la responsabilite de `validators.py`. Cette separation est volontaire : les operateurs de voisinage doivent rester rapides et travailler sur la matrice de couts deja consolidee.


In [4]:
r0 = routes[0]
print("route r1 :", r0)
print("cout r1  :", round(route_cost(r0, si.cost_matrix), 2))
print("charge r1:", route_load(r0, si.demands))
print("admissible ?", is_feasible(routes, si.demands, si.vehicle_capacity))


route r1 : [0, 2, 11, 1, 7, 3, 5, 4, 9, 0]
cout r1  : 364.35
charge r1: 40
admissible ? True


## 4. `two_opt` intra-tournee

**Principe.** 2-opt [2] est l'operateur fondateur des heuristiques de voisinage pour le TSP : on supprime deux aretes non adjacentes `(a,b)` et `(c,d)` d'une tournee, et on les reconnecte en `(a,c)` et `(b,d)`. Ca revient geometriquement a **inverser le segment** `[b...c]` entre les deux coupes. L'operation est tres locale (seules 2 aretes changent) mais a un effet global puissant : elle **denoue les croisements** geometriques, qui sont les principales sources d'inefficacite d'une tournee.

**Pourquoi seulement intra ?** Parce que 2-opt change l'ordre des clients **a l'interieur** d'une route mais ne change ni la composition ni la charge des routes. C'est l'operateur d'**intensification pure** : il polit chaque tournee individuellement. Pour reorganiser **entre** routes, on utilise relocate et swap (sections suivantes).

**Best-improvement vs first-improvement.** L'implementation parcourt **toutes** les paires `(i, j)` et applique le **meilleur** gain trouve avant de recommencer (best-improvement). L'alternative first-improvement appliquerait le **premier** gain rencontre. Best-improvement converge en moins d'iterations mais chaque iteration coute plus cher (O(n^2) evaluations contre O(n) en moyenne). Sur des petites tournees comme ici, la difference est negligeable ; sur de tres grandes tournees, first-improvement devient avantageux (Aarts & Lenstra [1, chap. 5]). On a choisi best-improvement pour la simplicite et la reproductibilite (un seul mouvement bien defini par iteration).

**Critere strict** `delta < -1e-9` : on n'accepte que les ameliorations **strictement** positives, pour eviter les boucles infinies sur des plateaux numeriques en virgule flottante.

`two_opt_intra` n'est qu'un wrapper qui applique `two_opt` a chaque tournee independamment.


In [5]:
before = total_cost(routes, si.cost_matrix)
after_routes = two_opt_intra(routes, si.cost_matrix)
after = total_cost(after_routes, si.cost_matrix)
print(f"avant 2-opt : {before:.2f}")
print(f"apres 2-opt : {after:.2f}   (gain {before - after:.2f})")


avant 2-opt : 1002.17
apres 2-opt : 906.00   (gain 96.17)


## 5. `relocate_inter` : deplacer un client entre tournees

**Principe.** On retire un client de sa tournee actuelle et on l'insere a une position d'une **autre** tournee. C'est le mouvement **inter-tournees** le plus simple et le plus puissant pour rééquilibrer les charges. Il est decrit comme un cas particulier de l'**Or-opt** introduit par Or [3] (qui generalise au deplacement de segments de 1, 2 ou 3 clients).

**Pourquoi inter et pas intra ?** Parce que **deplacer un client a l'interieur de sa propre tournee** est un cas degenere de 2-opt (il suffit d'inverser le segment approprie pour produire le meme effet) ; il serait redondant. La version inter, en revanche, est la seule facon de modifier la **composition** des tournees - c'est ce qui permet de fusionner deux petites routes ou d'eclater une grosse.

**Contrainte de capacite.** Avant tout test, on verifie que la tournee cible peut absorber la demande du client deplace. C'est une condition necessaire pour que la solution reste admissible.

**Best-improvement et nettoyage.** Comme 2-opt, on parcourt toutes les paires `(client, position cible)` et on applique le meilleur gain. Apres l'application, `_prune_empty_routes` supprime les tournees devenues vides (qui ne contiennent plus que `[depot, depot]`) - sinon elles polluent les iterations suivantes et le calcul du nombre de vehicules utilises.


In [6]:
before = total_cost(routes, si.cost_matrix)
relocated = relocate_inter(routes, si.cost_matrix, si.demands, si.vehicle_capacity)
after = total_cost(relocated, si.cost_matrix)
print(f"avant relocate : {before:.2f}   ({len(routes)} tournees)")
print(f"apres relocate : {after:.2f}   ({len(relocated)} tournees)")


avant relocate : 1002.17   (2 tournees)
apres relocate : 769.86   (2 tournees)


## 6. `swap_inter` : echanger deux clients entre tournees

**Principe.** On choisit un client `c1` dans une tournee et un client `c2` dans une autre, et on les **echange**. Contrairement a `relocate_inter` qui peut creer ou supprimer des tournees, swap **conserve le nombre de tournees** et leur taille (chaque route perd et gagne exactement un client).

**Pourquoi avoir les deux ?** Parce qu'ils explorent des voisinages **complementaires** :

- **Relocate** modifie les charges des deux tournees concernees -> utile pour rééquilibrer.
- **Swap** garde les charges quasi-stables (variation de `demand(c2) - demand(c1)`) -> utile pour reorganiser la geometrie sans perturber l'equilibre. C'est particulierement efficace quand deux tournees de charges similaires ont des clients mal places relativement.

Beaucoup de mouvements atteignables par swap ne le sont **pas** par une seule application de relocate (il en faudrait deux successives, chacune potentiellement degradante). C'est pourquoi swap est un complement classique a relocate dans toutes les heuristiques VRP modernes (Cordeau et al. [4, section 4.2]).

**Contraintes de capacite.** Les deux directions sont verifiees : `loads[r1] - d1 + d2 <= capacity` et symmetrique. Sinon le mouvement serait inadmissible.


In [7]:
swapped = swap_inter(relocated, si.cost_matrix, si.demands, si.vehicle_capacity)
print(f"apres swap     : {total_cost(swapped, si.cost_matrix):.2f}   ({len(swapped)} tournees)")


apres swap     : 769.86   (2 tournees)


## 7. `local_search` : composition jusqu'a stabilite

**Principe : VND** (*Variable Neighborhood Descent*, descente a voisinages variables, Mladenovic & Hansen [5]). On enchaine les operateurs dans un ordre fixe : `relocate_inter -> swap_inter -> two_opt_intra`. Si un operateur ameliore, on **recommence depuis le debut** ; si aucun n'ameliore, on a atteint un optimum local **simultane** des trois voisinages et on s'arrete.

**Pourquoi cet ordre ?** Convention pratique : du plus impactant (relocate change la composition des routes) au plus local (2-opt polit l'interieur). Recommencer depuis le debut apres chaque amelioration peut paraitre lent, mais c'est la garantie d'**etre au fond** : un swap qui suit un relocate peut debloquer un nouveau relocate. Cette structure est detaillee comme la variante de base de VND par Hansen, Mladenovic, Brimberg & Perez [6, section 2].

**Optimum local au sens du voisinage compose.** Une solution rendue par `local_search` n'est pas seulement un optimum de chaque operateur pris isolement - c'est un optimum **simultane** des trois. Cette propriete est plus forte qu'une descente single-neighborhood et explique pourquoi `local_search` est utilisable telle quelle comme polissage final pour SA, tabou et GRASP, et comme phase memetique du GA.

**Critere strict.** L'arret se fait quand le cout n'a pas baisse strictement (`< -1e-9`). Encore une fois, c'est une protection contre les boucles infinies sur des plateaux numeriques en virgule flottante.


In [8]:
ls_routes = local_search(routes, si.cost_matrix, si.demands, si.vehicle_capacity)
print(f"cout initial    : {total_cost(routes, si.cost_matrix):.2f}")
print(f"apres local_search : {total_cost(ls_routes, si.cost_matrix):.2f}")
print(f"nombre de tournees : {len(ls_routes)}")


cout initial    : 1002.17
apres local_search : 769.86
nombre de tournees : 2


## 8. Voisins aleatoires

**Pourquoi des versions aleatoires ?** Le recuit simule fonctionne sur le principe de **tirer un voisin au hasard** (et pas le meilleur), pour deux raisons :

1. **Performance par iteration** : tirer un voisin coute O(1), balayer tout le voisinage coute O(n^2). SA fait des milliers d'iterations, donc l'economie est massive.
2. **Diversification** : le critere de Metropolis a besoin d'un voisin **non biaise** vers les ameliorations, sinon la probabilite d'accepter une degradation devient mal calibree par rapport au paysage de cout.

`random_neighbor` choisit **uniformement** entre `random_relocate`, `random_swap` et `random_two_opt`. Cette diversite d'operateurs - le **voisinage compose** au sens de Hansen et al. [6] - est essentielle : utiliser un seul operateur creerait des biais (par exemple, 2-opt seul ne peut jamais changer la composition des routes, donc SA ne pourrait pas rééquilibrer les charges).

**Gestion du cas vide.** Chaque `random_*` peut renvoyer `None` quand le tirage ne produit aucun voisin admissible (par ex. swap entre deux clients quand toutes les paires violeraient la capacite). `random_neighbor` retombe alors sur une **copie de la solution courante**. Cette copie de fallback est preferable a `None` cote SA, qui peut ainsi continuer sa boucle sans cas particulier - le cout du voisin "vide" est egal au cout courant, donc `delta = 0`, donc accepte automatiquement, donc l'iteration est juste sans effet.


In [9]:
rng = random.Random(42)
for step in range(6):
    neighbor = random_neighbor(ls_routes, si.demands, si.vehicle_capacity, rng)
    c = total_cost(neighbor, si.cost_matrix)
    print(f"voisin {step + 1} : cout={c:.2f} ; {len(neighbor)} tournees")


voisin 1 : cout=830.21 ; 2 tournees
voisin 2 : cout=919.04 ; 2 tournees
voisin 3 : cout=838.74 ; 2 tournees
voisin 4 : cout=935.86 ; 2 tournees
voisin 5 : cout=785.67 ; 2 tournees
voisin 6 : cout=836.95 ; 2 tournees


## 9. Reproductibilite

Deux familles distinctes :

- **Operateurs deterministes** (`relocate_inter`, `swap_inter`, `two_opt`, `local_search`) : strictement reproductibles par construction. Meme entree -> meme sortie, sans aucun parametre aleatoire.
- **Operateurs aleatoires** (`random_*`, `random_neighbor`) : prennent une `random.Random` **passee explicitement** en argument. Ils ne touchent jamais le `random` global de Python. Consequence : deux executions avec la meme graine produisent **exactement** la meme trajectoire de voisins.

Cette discipline est cruciale pour debugger les metaheuristiques (SA et GA en particulier) ou pour comparer plusieurs algorithmes a chance egale dans `benchmark.py`. Elle suit la regle generale du projet : **un seul `random.Random(seed)` cree par algorithme**, jamais d'utilisation du module `random` global.

---

## References

[1] **Aarts, E. & Lenstra, J. K. (eds.) (2003).** *Local Search in Combinatorial Optimization.* Princeton University Press. https://doi.org/10.1515/9780691187563

[2] **Lin, S. (1965).** *Computer solutions of the traveling salesman problem.* Bell System Technical Journal, 44(10), 2245-2269. https://doi.org/10.1002/j.1538-7305.1965.tb04146.x

[3] **Or, I. (1976).** *Traveling Salesman-Type Combinatorial Problems and Their Relation to the Logistics of Regional Blood Banking.* PhD thesis, Northwestern University.

[4] **Cordeau, J.-F., Gendreau, M., Laporte, G., Potvin, J.-Y. & Semet, F. (2002).** *A guide to vehicle routing heuristics.* Journal of the Operational Research Society, 53(5), 512-522. https://doi.org/10.1057/palgrave.jors.2601319

[5] **Mladenovic, N. & Hansen, P. (1997).** *Variable neighborhood search.* Computers & Operations Research, 24(11), 1097-1100. https://doi.org/10.1016/S0305-0548(97)00031-2

[6] **Hansen, P., Mladenovic, N., Brimberg, J. & Perez, J. A. M. (2019).** *Variable Neighborhood Search.* In Gendreau M. & Potvin J.-Y. (eds.), Handbook of Metaheuristics (3rd ed.), 57-97. Springer. https://doi.org/10.1007/978-3-319-91086-4_3


In [10]:
def trajectory(seed: int) -> list[float]:
    rng = random.Random(seed)
    s = [route[:] for route in ls_routes]
    out = []
    for _ in range(5):
        s = random_neighbor(s, si.demands, si.vehicle_capacity, rng)
        out.append(round(total_cost(s, si.cost_matrix), 2))
    return out

print("seed=1 :", trajectory(1))
print("seed=1 :", trajectory(1))
print("seed=2 :", trajectory(2))


seed=1 : [879.28, 996.22, 1128.58, 1063.31, 1063.31]
seed=1 : [879.28, 996.22, 1128.58, 1063.31, 1063.31]
seed=2 : [805.17, 921.73, 913.2, 972.24, 1032.62]
